# 04. Telegram-коннектор

Добавляем реальный внешний сервис — Telegram Bot API.
В workshop используем `dry_run=True`, чтобы показать payload без отправки.

Генерируется `agents/step_04_telegram.py`.

```bash
uv run langgraph dev --config langgraph.steps.json
```

## Визуальная рамка

![Telegram dry-run connector](visuals/04_telegram_dry_run.svg)

Этот шаг соответствует production-паттерну safety: Telegram подключен как реальный канал, но live demo сначала показывает payload в `dry_run=True`.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend, LocalShellBackend
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'pyproject.toml').exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

from connectors.demo import DEMO_TOOLS
from connectors.telegram import TELEGRAM_TOOLS, send_telegram_message

# Пробуем Telegram tool в dry-run
print(send_telegram_message.invoke({
    "message": "OpenClaw connectors are working.",
    "dry_run": True,
}))

Теперь соберём агента с demo + telegram tools.

In [ ]:
DEFAULT_MODEL = 'openrouter:tencent/hy3:free'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)


def workspace_root() -> Path:
    return Path(os.getenv('OPENCLAW_WORKSPACE', '.')).expanduser().resolve()


def backend():
    root_dir = workspace_root()
    if os.getenv('OPENCLAW_ENABLE_LOCAL_SHELL') != '1':
        return FilesystemBackend(root_dir=root_dir, virtual_mode=True)
    return LocalShellBackend(
        root_dir=root_dir, virtual_mode=True,
        inherit_env=True, timeout=120, max_output_bytes=80_000,
    )


BASE_PROMPT = """\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.
"""


agent = create_deep_agent(
    model=model_name(),
    tools=[*DEMO_TOOLS, *TELEGRAM_TOOLS],
    system_prompt=BASE_PROMPT,
    backend=backend(),
    interrupt_on={"execute": True},
)

print(type(agent).__name__)

### Сгенерировать entrypoint

In [ ]:
import json

AGENTS_DIR = REPO_ROOT / "agents"
AGENTS_DIR.mkdir(exist_ok=True)

agent_code = '''"""Step 04: adds Telegram Bot API connector (dry-run by default)."""

from __future__ import annotations

import os
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend, LocalShellBackend
from dotenv import load_dotenv

from connectors.demo import DEMO_TOOLS
from connectors.telegram import TELEGRAM_TOOLS

load_dotenv()

DEFAULT_MODEL = "openrouter:tencent/hy3:free"


def _workspace_root() -> Path:
    return Path(os.getenv("OPENCLAW_WORKSPACE", ".")).expanduser().resolve()


def _backend():
    root_dir = _workspace_root()
    if os.getenv("OPENCLAW_ENABLE_LOCAL_SHELL") != "1":
        return FilesystemBackend(root_dir=root_dir, virtual_mode=True)
    return LocalShellBackend(
        root_dir=root_dir, virtual_mode=True,
        inherit_env=True, timeout=120, max_output_bytes=80_000,
    )


agent = create_deep_agent(
    model=os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL),
    tools=[*DEMO_TOOLS, *TELEGRAM_TOOLS],
    system_prompt="""\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.
""",
    backend=_backend(),
    interrupt_on={"execute": True},
)
'''

step_file = AGENTS_DIR / "step_04_telegram.py"
step_file.write_text(agent_code)

steps_config = {
    "dependencies": ["."],
    "graphs": {
        "openclaw": "./agents/step_04_telegram.py:agent",
    },
    "env": ".env",
}
config_path = REPO_ROOT / "langgraph.steps.json"
config_path.write_text(json.dumps(steps_config, indent=2) + "\n")

print(f"Wrote {step_file}")
print(f"Wrote {config_path}")
print()
print("Restart:")
print("  uv run langgraph dev --config langgraph.steps.json")

### Проверочный prompt

После перезапуска попробуй:

> "Use the Telegram connector in dry-run mode. Prepare a short workshop message that says OpenClaw connectors are working."